In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'src' / 'model.py').exists(), 'Abrir a pasta do projeto no VS Code.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.model import Problem, Request, Vehicle, State, Stop, Route, example_route
from src.analysis import audit, individual_feasibility
problem = Problem(ROOT / 'data_tiny')


In [ ]:
display(pd.DataFrame([{'tabela':n,'registos':len(d),'colunas':len(d.columns)} for n,d in problem.tables.items()]))
for name in ['orders','order_items','vehicles','locations']:
    print(name)
    display(problem.tables[name].head())

In [ ]:
display(problem.tables['rules'])

In [ ]:
checks = audit(problem)
display(checks)
print('Total de problemas detetados:', checks['problemas'].sum())
display(checks[checks['problemas'] > 0])

In [ ]:
items=problem.tables['order_items'].copy()
items['dia']=pd.to_datetime(items.delivery_start).dt.date
display(items.groupby('dia').agg(itens=('order_item_id','size'),peso_kg=('weight_kg','sum')))
display(problem.tables['vehicles'][['vehicle_id','capacity_kg','refrigerated']])
print('Peso total (kg):', round(items.weight_kg.sum(),2))
print('Maior item (kg):', items.weight_kg.max())
print('Capacidade conjunta (kg):',sum(v.capacity for v in problem.vehicles.values()))
ax=items.groupby('dia').weight_kg.sum().plot.bar(color='#327a91',rot=0,figsize=(7,4))
ax.set(xlabel='Dia de entrega',ylabel='Peso (kg)',title='Procura por dia')
plt.tight_layout();plt.show()

In [ ]:
loc=problem.tables['locations']
fig,ax=plt.subplots(figsize=(9,6))
for typ,g in loc.groupby('location_type'):
    ax.scatter(g.longitude,g.latitude,label=typ,s=90)
for (lon,lat),group in loc.groupby(['longitude','latitude']):
    label=', '.join(str(i) for i in group.location_id)
    ax.annotate(label,(lon,lat),xytext=(5,7),textcoords='offset points')
ax.set(xlabel='Longitude',ylabel='Latitude',title='Paragens do cenário tiny')
ax.set_aspect(1 / __import__('math').cos(__import__('math').radians(loc.latitude.mean())))
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()
display(loc[['location_id','location_type','name','municipality']])

In [ ]:
individual=individual_feasibility(problem)
display(individual)
print('Itens viáveis isoladamente:', int(individual.individual_feasible.sum()), '/',len(individual))
display(individual.loc[~individual.individual_feasible])

In [ ]:
small=Problem(ROOT / 'data_small')
small_checks=audit(small)
print('Small:',len(small.requests),'itens;',len(small.vehicles),'veículos')
display(small_checks[small_checks['problemas']>0])

In [ ]:
print(f'O tiny contém {len(problem.requests)} itens, {len(problem.tables["orders"])} encomendas e {len(problem.vehicles)} veículos.')
print(f'A auditoria contabilizou {checks.problemas.sum()} problemas estruturais nos testes implementados.')
print(f'{(~individual.individual_feasible).sum()} itens não passaram o teste individual.')
print('A viabilidade simultânea e a qualidade das rotas serão estudadas na fase dos algoritmos.')